In [1]:
import os  # Filesystem checks
import numpy as np  # Arrays and NaNs
import pandas as pd  # DataFrame IO and ops
from sklearn.preprocessing import StandardScaler  # Z-score scaling
from joblib import load  # Load saved scaler
from typing import List, Tuple, Dict, Any  # Type hints

In [2]:
df_cl = pd.read_csv('class_df_all_with_filtered_rdkit_features.csv')

In [3]:
df_cl.shape

(781179, 122)

In [4]:
df_cl.head()

,number,SMILES,Name,Affinity,label,MaxEStateIndex,MinEStateIndex,MinAbsEStateIndex,qed,MolWt,...,fr_piperzine,fr_priamide,fr_pyridine,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_unbrch_alkane,fr_urea
0,1,N=c1nc2n(cc1F)[C@H]1O[C@H](CO)[C@@H](O)[C@H]1O2,ZINC000000001477,-6.1,1,13.213502,-1.010236,0.029481,0.557798,243.194,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,Cc1cn([C@H]2C[C@@H](CO)N(O)C2)c(=O)[nH]c1=O,ZINC000000004266,-5.9,0,11.625685,-0.488343,0.162613,0.606489,241.247,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,CC(=O)N[C@@H]1C[C@@H](O)[C@H](CO)O[C@@H]1O,ZINC000000005637,-5.0,0,10.706467,-1.181065,0.172593,0.415739,205.210,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4,Nc1ncc2c(ncn2COC(CO)CO)n1,ZINC000000005980,-5.3,0,8.853217,-0.609120,0.148378,0.595031,239.235,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5,Cc1cn([C@H]2O[C@@H](CO)[C@@H]2CO)c(=O)[nH]c1=O,ZINC000000006018,-5.7,0,11.566241,-0.638704,0.193941,0.588358,242.231,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [5]:
df_cl.tail()

,number,SMILES,Name,Affinity,label,MaxEStateIndex,MinEStateIndex,MinAbsEStateIndex,qed,MolWt,...,fr_piperzine,fr_priamide,fr_pyridine,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_unbrch_alkane,fr_urea
781174,782193,COC[C@@H]1[C@H](NC(CO)CO)[C@@H]2CCO[C@H]12,ZINC000218404185,NaN,0,9.066176,-0.230642,0.036760,0.545618,231.292,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
781175,782194,CN(C)CCO[C@@H]1COCC[C@H]1NC(=O)CO,ZINC000218743468,NaN,0,11.141895,-0.484350,0.064294,0.617836,246.307,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
781176,782195,Cn1cc(S(=O)(=O)F)c(=O)n(C)c1=O,ZINC000238857165,NaN,0,12.517260,-5.088310,0.520301,0.555618,222.197,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
781177,782196,O=C1NC(=O)[C@@H](CCS(=O)(=O)F)N1,ZINC000307689379,NaN,0,11.994428,-4.583802,0.253565,0.455746,210.186,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
781178,782197,O=C1NC(=O)[C@H](CCS(=O)(=O)F)N1,ZINC000307689380,NaN,0,11.994428,-4.583802,0.253565,0.455746,210.186,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [6]:
df_all = df_cl

In [7]:
df = df_cl

In [8]:
# drop duplicates and creating desc_df


# 1.  Compute the mean label for every SMILES (handles duplicates)
mean_label = (
    df.groupby('SMILES', as_index=False)['Affinity']  # group by SMILES
      .mean()                                      # average label
      .set_index('SMILES')['Affinity']                # Series: index = SMILES, value = mean label
)


# 2.  Drop duplicate SMILES but KEEP FIRST ROW to preserve original order & features
df_dedup = df.drop_duplicates(subset='SMILES', keep='first').reset_index(drop=True)


# 3.  Replace the kept rows’ label with the mean from step-1
df_dedup['Affinity'] = df_dedup['SMILES'].map(mean_label)


# 4.  Move ‘label’ to be the 4-th column (index position 3)
label_col = df_dedup.pop('Affinity')      # remove
df_dedup.insert(3, 'Affinity', label_col) # re-insert at index 3

print(f"Deduplicated DataFrame shape: {df_dedup.shape}")
df = df_dedup
# (Optional) save
# df_dedup.to_csv('deduplicated_dataset.csv', index=False)


# 5.  Build desc_df by dropping the first 4 columns (metadata + label)
desc_df = df_dedup.iloc[:, 5:].copy()  # only numerical descriptor columns remain
print(f"desc_df shape: {desc_df.shape}")


Deduplicated DataFrame shape: (776427, 122)
desc_df shape: (776427, 117)


In [9]:
desc_df.head()

,MaxEStateIndex,MinEStateIndex,MinAbsEStateIndex,qed,MolWt,FpDensityMorgan1,BCUT2D_MWHI,BCUT2D_MWLOW,BCUT2D_CHGHI,BCUT2D_CHGLO,...,fr_piperzine,fr_priamide,fr_pyridine,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_unbrch_alkane,fr_urea
0,13.213502,-1.010236,0.029481,0.557798,243.194,1.588235,19.142206,10.132514,2.505182,-2.136804,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,11.625685,-0.488343,0.162613,0.606489,241.247,1.529412,16.502492,10.153728,2.318689,-2.128119,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,10.706467,-1.181065,0.172593,0.415739,205.210,1.571429,16.624561,10.006924,2.355840,-2.357681,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,8.853217,-0.609120,0.148378,0.595031,239.235,1.352941,16.516978,10.402126,2.085244,-2.124338,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,11.566241,-0.638704,0.193941,0.588358,242.231,1.411765,16.550218,9.951202,2.428104,-2.429558,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [10]:
# To have desc_df + duplicates being used in final prediction
desc_df_all = df_all.iloc[:, 5:].copy()  
print(f"desc_df shape: {desc_df_all.shape}")
desc_df_all.head()

desc_df shape: (781179, 117)


,MaxEStateIndex,MinEStateIndex,MinAbsEStateIndex,qed,MolWt,FpDensityMorgan1,BCUT2D_MWHI,BCUT2D_MWLOW,BCUT2D_CHGHI,BCUT2D_CHGLO,...,fr_piperzine,fr_priamide,fr_pyridine,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_unbrch_alkane,fr_urea
0,13.213502,-1.010236,0.029481,0.557798,243.194,1.588235,19.142206,10.132514,2.505182,-2.136804,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,11.625685,-0.488343,0.162613,0.606489,241.247,1.529412,16.502492,10.153728,2.318689,-2.128119,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,10.706467,-1.181065,0.172593,0.415739,205.210,1.571429,16.624561,10.006924,2.355840,-2.357681,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,8.853217,-0.609120,0.148378,0.595031,239.235,1.352941,16.516978,10.402126,2.085244,-2.124338,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,11.566241,-0.638704,0.193941,0.588358,242.231,1.411765,16.550218,9.951202,2.428104,-2.429558,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [11]:
# Handle NaNs in target 'Affinity' column to ignore nonlabeled data.

# 1) Build a boolean mask on df for rows with non-missing Affinity
mask_aff = df['Affinity'].notna()  # True for rows to keep

# 2) Apply the same mask to BOTH df and desc_df before any reset_index
df = df.loc[mask_aff].copy()                # Keep rows with Affinity present
desc_df = desc_df.loc[mask_aff.values].copy()  # Same rows by position

# 3) Now reset indexes together so they stay aligned
df = df.reset_index(drop=True)
desc_df = desc_df.reset_index(drop=True)

print(df.shape, desc_df.shape)


(9989, 122) (9989, 117)


In [12]:
df.head()

,number,SMILES,Name,Affinity,label,MaxEStateIndex,MinEStateIndex,MinAbsEStateIndex,qed,MolWt,...,fr_piperzine,fr_priamide,fr_pyridine,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_unbrch_alkane,fr_urea
0,1,N=c1nc2n(cc1F)[C@H]1O[C@H](CO)[C@@H](O)[C@H]1O2,ZINC000000001477,-6.1,1,13.213502,-1.010236,0.029481,0.557798,243.194,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,Cc1cn([C@H]2C[C@@H](CO)N(O)C2)c(=O)[nH]c1=O,ZINC000000004266,-5.9,0,11.625685,-0.488343,0.162613,0.606489,241.247,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,CC(=O)N[C@@H]1C[C@@H](O)[C@H](CO)O[C@@H]1O,ZINC000000005637,-5.0,0,10.706467,-1.181065,0.172593,0.415739,205.210,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4,Nc1ncc2c(ncn2COC(CO)CO)n1,ZINC000000005980,-5.3,0,8.853217,-0.609120,0.148378,0.595031,239.235,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5,Cc1cn([C@H]2O[C@@H](CO)[C@@H]2CO)c(=O)[nH]c1=O,ZINC000000006018,-5.7,0,11.566241,-0.638704,0.193941,0.588358,242.231,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [13]:
df.shape

(9989, 122)

In [ ]:
# =========================
# PHASE 1: PREPARE DESCRIPTOR SPACE 
# Assumes the following are already defined from your steps:
# - df, desc_df: deduplicated, Affinity-notna filtered training subset and its feature matrix
# - df_all, desc_df_all: full dataset (may include unlabeled) and its feature matrix
# - You have a saved StandardScaler from model training (recommended to reuse)
# =========================

import numpy as np                  # Numerical ops
import pandas as pd                 # Data handling
from joblib import load             # Load saved scaler
from sklearn.preprocessing import StandardScaler  # Type hints and IDE help

# -------------------------
# Section A: Configure paths
# -------------------------
SCALER_PATH = "scaler_LogisticRegression.joblib"  

# -----------------------------------------
# Section B: Align and sanitize feature data
# -----------------------------------------
# Use the exact feature list present in desc_df (training features after your filtering)
feature_cols = list(desc_df.columns)  # Ordered list of descriptor columns

# Replace +/- inf with NaN for both train and all sets
desc_df = desc_df.replace([np.inf, -np.inf], np.nan)
desc_df_all = desc_df_all.replace([np.inf, -np.inf], np.nan)

# Ensure the "all" matrix has at least these training columns; add missing with NaN to surface issues
for c in feature_cols:
    if c not in desc_df_all.columns:
        desc_df_all[c] = np.nan  # Create missing columns to maintain column order

# Reorder the "all" matrix to match training feature order exactly
desc_df_all = desc_df_all[feature_cols]

# Quick sanity checks (fail fast if unexpected NaNs remain)
if desc_df.isna().any().any():
    raise ValueError("NaNs detected in desc_df features; resolve upstream to match original training preprocessing.")
if desc_df_all[feature_cols].isna().any().any():
    # Allowing unlabeled rows is fine, but feature NaNs are not compatible with a fitted StandardScaler
    raise ValueError("NaNs detected in desc_df_all features; ensure preprocessing matches training before scaling.")

# --------------------------------------
# Section C: Load saved scaler and apply
# --------------------------------------
# Load the scaler used during model training (do NOT refit for AD)
scaler: StandardScaler = load(SCALER_PATH)  # Reuse training means/variances for exact space

# Extract numpy arrays in the fixed feature order
X_train = desc_df[feature_cols].values      # Training features (Affinity-notna subset)
X_all = desc_df_all[feature_cols].values    # All features (for later AD labeling)

# Transform into standardized space (z-scores)
X_train_std = scaler.transform(X_train)     # Standardized training matrix (for AD calibration)
X_all_std = scaler.transform(X_all)         # Standardized full matrix (for later AD labels)

# --------------------------------------
# Section D: Pack helpful artifacts
# --------------------------------------
# Keep standardized matrices and aligned DataFrames (optional but convenient)
X_train_std_df = pd.DataFrame(X_train_std, index=df.index, columns=feature_cols)      # Train z-scores
X_all_std_df = pd.DataFrame(X_all_std, index=df_all.index, columns=feature_cols)      # All z-scores

# Print quick shapes to confirm alignment
print("Phase 1 complete.")
print("Training (Affinity-notna) standardized shape:", X_train_std.shape)
print("All standardized shape:", X_all_std.shape)

# --------------------------------------
# Outputs you will use in next phases:
# - X_train_std (numpy array): standardized training features for k-NN distance distribution
# - X_all_std   (numpy array): standardized full set for per-molecule AD labeling later
# - feature_cols (list): exact feature order used for scaling/distances
# - scaler (StandardScaler): saved scaler reused for consistent distance space
# --------------------------------------


Phase 1 complete.
Training (Affinity-notna) standardized shape: (9989, 117)
All standardized shape: (781179, 117)


In [15]:
# =========================
# PHASE 2: CALIBRATE AD THRESHOLD (k-NN distance on training set, optional Mondrian and leverage)
# Prereqs in memory from Phase 1: df (Affinity-notna training subset), X_train_std (z-scored training features), feature_cols (feature names)
# =========================

import numpy as np  # Numerical operations for distances, percentiles, and linear algebra [web:113]
import pandas as pd  # Tabular packaging of k-NN distances and AD flags [web:113]
from sklearn.neighbors import NearestNeighbors  # Efficient k-NN search for distance-based AD calibration [web:113]

# -------------------------
# Section A: k-NN distances
# -------------------------
k = 25  # Neighborhood size; 5–50 is typical for ~10k samples to balance local density and robustness [web:2]
metric = "euclidean"  # Euclidean on standardized features is standard for descriptor-space AD [web:2]

# Fit neighbor index on training set standardized descriptors
nn = NearestNeighbors(n_neighbors=k + 1, metric=metric, n_jobs=-1)  # Request k+1 to include the point itself at distance 0 [web:113]
nn.fit(X_train_std)  # Build the neighbor structure once on the training matrix [web:113]

# Query neighbors for each training point against the training set
dists, idxs = nn.kneighbors(X_train_std, return_distance=True)  # Shape: (n_train, k+1), first neighbor is self [web:113]

# Remove self-neighbor (distance 0) and aggregate k-NN distance per sample
knn_dists_k = dists[:, 1:]  # Exclude the self-distance at column 0 to avoid biasing the mean [web:113]
knn_dist_mean = knn_dists_k.mean(axis=1)  # Mean k-NN distance as a continuous density proxy [web:2]
knn_dist_median = np.median(knn_dists_k, axis=1)  # Median can be more robust in skewed local densities [web:2]

# -----------------------------------
# Section B: Global AD cutoff by quantile
# -----------------------------------
q = 95.0  # Choose the global in-domain quantile; 95th percentile is a common heuristic in QSAR AD [web:2]
cutoff_mean_q = np.percentile(knn_dist_mean, q)  # Global cutoff on the mean k-NN distance distribution [web:2]
cutoff_median_q = np.percentile(knn_dist_median, q)  # Same idea for the median-based summary [web:2]

# Build calibration DataFrame on the training set
ad_calib = pd.DataFrame({
    "k": k,  # Store neighborhood size for transparency and reproducibility [web:41]
    "metric": metric,  # Record the distance metric used [web:41]
    "kNN_mean": knn_dist_mean,  # Per-sample mean k-NN distance over k neighbors [web:2]
    "kNN_median": knn_dist_median,  # Per-sample median k-NN distance over k neighbors [web:2]
    "cutoff_mean_q": cutoff_mean_q,  # Global 95th percentile cutoff (mean) [web:2]
    "cutoff_median_q": cutoff_median_q,  # Global 95th percentile cutoff (median) [web:2]
}, index=df.index)  # Align rows with df’s training subset for later merges [web:2]

# In-domain flags under global rule
ad_calib["inAD_mean_q"] = ad_calib["kNN_mean"] <= ad_calib["cutoff_mean_q"]  # True if within global mean-based cutoff [web:2]
ad_calib["inAD_median_q"] = ad_calib["kNN_median"] <= ad_calib["cutoff_median_q"]  # True if within global median-based cutoff [web:2]

# -----------------------------------
# Section C: Mondrian (class-conditional) cutoffs (optional)
# -----------------------------------
if "Label" in df.columns:  # If class labels are available, compute class-conditional thresholds to mitigate class imbalance and asymmetry [web:2]
    mondrian_cut_mean = {}  # Per-class mean-distance cutoff at the same quantile q [web:2]
    mondrian_cut_median = {}  # Per-class median-distance cutoff at the same quantile q [web:2]
    for cls in sorted(df["Label"].unique()):  # Iterate classes present in the training subset [web:2]
        cls_mask = df["Label"].values == cls  # Boolean mask for current class [web:2]
        mondrian_cut_mean[cls] = np.percentile(knn_dist_mean[cls_mask], q)  # Class-conditional 95th percentile for mean distance [web:2]
        mondrian_cut_median[cls] = np.percentile(knn_dist_median[cls_mask], q)  # Class-conditional 95th percentile for median distance [web:2]
    ad_calib["cutoff_mean_q_mondrian"] = df["Label"].map(mondrian_cut_mean)  # Map class-specific mean cutoff per row [web:2]
    ad_calib["cutoff_median_q_mondrian"] = df["Label"].map(mondrian_cut_median)  # Map class-specific median cutoff per row [web:2]
    ad_calib["inAD_mean_q_mondrian"] = ad_calib["kNN_mean"] <= ad_calib["cutoff_mean_q_mondrian"]  # In-AD flag under class-conditional mean cutoff [web:2]
    ad_calib["inAD_median_q_mondrian"] = ad_calib["kNN_median"] <= ad_calib["cutoff_median_q_mondrian"]  # In-AD flag under class-conditional median cutoff [web:2]

# -----------------------------------
# Section D: Optional leverage check (hat values) for linear extrapolation flags
# -----------------------------------
# Compute classical hat matrix diagonal h = diag(X (X^T X)^{-1} X^T) on standardized features with intercept, and set warning h* = 3(p+1)/n [web:117]
X_aug = np.hstack([np.ones((X_train_std.shape[0], 1)), X_train_std])  # Add intercept to count p+1 parameters in leverage [web:117]
XtX = X_aug.T @ X_aug  # Cross-product to build normal equations matrix [web:117]
XtX_inv = np.linalg.pinv(XtX)  # Pseudo-inverse for numerical stability in high dimensions [web:117]
leverages = np.einsum("ij,jk,ik->i", X_aug, XtX_inv, X_aug)  # Efficient diagonal extraction without forming full hat matrix [web:117]
p_eff = X_aug.shape[1]  # Number of parameters including intercept for leverage threshold [web:117]
n_train = X_train_std.shape[0]  # Training sample size for h* computation [web:117]
h_star = 3.0 * p_eff / n_train  # Williams’ warning leverage threshold commonly used in QSAR AD [web:117]

# Attach leverage and leverage-based flag
ad_calib["leverage_h"] = leverages  # Store per-sample leverage for Williams plot diagnostics [web:117]
ad_calib["inAD_leverage"] = ad_calib["leverage_h"] <= h_star  # Flag low-extrapolation region under h* [web:117]
ad_calib["h_star"] = h_star  # Record the threshold used for transparency [web:117]

# -----------------------------------
# Section E: Merge back for reporting and quick sanity prints
# -----------------------------------
df_ad = df.join(ad_calib)  # Combine training metadata with AD distances and flags for downstream analysis [web:2]

# Print concise calibration summary
print(f"k={k}, metric={metric}, n_train={n_train}")  # Core calibration settings for traceability [web:113]
print(f"Global cutoff (mean, {q}th pct): {cutoff_mean_q:.4f}")  # Global mean-distance threshold at chosen quantile [web:2]
print(f"Global cutoff (median, {q}th pct): {cutoff_median_q:.4f}")  # Global median-distance threshold at chosen quantile [web:2]
print(f"h* (leverage): {h_star:.6f}")  # Williams’ warning leverage threshold for structural extrapolation [web:117]

# df_ad now contains:
# - kNN_mean, kNN_median: continuous proximity measures (smaller = denser neighborhood) [web:2]
# - inAD_mean_q, inAD_median_q: global in-domain flags at the chosen quantile [web:2]
# - Optional Mondrian columns if Label exists: inAD_mean_q_mondrian, inAD_median_q_mondrian [web:2]
# - leverage_h, inAD_leverage, h_star: leverage-based extrapolation diagnostics [web:117]


k=25, metric=euclidean, n_train=9989
Global cutoff (mean, 95.0th pct): 11.6267
Global cutoff (median, 95.0th pct): 12.2814
h* (leverage): 0.035439


k=25 — neighborhood size used in k-NN. Each sample’s density summary is based on its 25 nearest neighbors (excluding itself). This is a moderate k suitable for ~10k samples (balances local sensitivity with robustness).

metric=euclidean — distance metric used in standardized descriptor space. Euclidean on z-scored features is standard because each feature contributes equally (zero mean, unit variance).

n_train=9989 — number of training samples used to build the neighbor index and compute thresholds.

Global cutoff (mean, 95.0th pct) = 11.6267
This means: the 95th percentile of the distribution of kNN_mean (the per-sample mean of 25 neighbor distances) is 11.6267. Any sample (train or future test) with a mean kNN distance greater than 11.6267 would be flagged out-of-domain under the global mean rule.

Global cutoff (median, 95.0th pct) = 12.2814
Similarly, the 95th percentile of kNN_median is 12.2814. Because median is sometimes larger for skewed neighborhoods, the median cutoff can be numerically different from the mean cutoff. A sample with kNN_median > 12.2814 is OOD by the median rule.

h (leverage) = 0.035439*
This is the Williams warning threshold computed as 3*(p+1)/n. If the per-sample leverage h_i exceeds 0.035439, the sample is considered structurally extrapolative (i.e., high influence / far from the centroid of descriptor space in a linear sense).

Leverage h_star

h_star is a heuristic threshold; it flags potential extrapolation. There’s no strict performance guarantee tied to it, but:
If very few samples exceed h_star → fine.
If many exceed → dimension p may be too large relative to n; consider feature reduction (PCA, feature selection) or increasing training data.
Typical action if many leverage warnings: remove or investigate high-leverage samples (could be data errors or structurally extreme compounds), or reduce effective dimensionality.

| Concept         | Example                                                                  | Reliability  |
| --------------- | ------------------------------------------------------------------------ | ------------ |
| ✅ Interpolation | Predicting activity for a molecule similar to many training molecules    | Usually good |
| ❌ Extrapolation | Predicting activity for a molecule with extreme or new descriptor values | Risky/poor   |

In [ ]:
# =========================
# PHASE 3: ASSIGN AD LABELS TO NEW MOLECULES (k-NN distance + leverage)
# Prereqs in memory from Phases 1–2:
# - X_train_std: standardized training features (Affinity-notna subset) 
# - X_all_std: standardized full feature matrix for all molecules to label (includes unlabeled) 
# - cutoff_mean_q, cutoff_median_q: calibrated global cutoffs from Phase 2 (e.g., 95th pct) 
# - k, metric: neighborhood size and distance metric used in Phase 2 (e.g., k=25, euclidean) 
# Optional in memory:
# - df_all: DataFrame with metadata (e.g., SMILES/Affinity/Predicted_Prob/Predicted_Label) to attach AD outputs 
# =========================

import numpy as np  # Arrays, einsum, and linear algebra ops 
import pandas as pd  # Join AD outputs to original DataFrame for reporting 
from sklearn.neighbors import NearestNeighbors  # k-NN distances to training set for AD 

# --------------------------------
# Section A: Build/query k-NN index
# --------------------------------
# Rebuild the neighbor index on the training set to query all molecules consistently (safe even if Phase 2's object was not kept) 
nn = NearestNeighbors(n_neighbors=k, metric=metric, n_jobs=-1)  # Use same k and metric as in calibration 
nn.fit(X_train_std)  # Fit once on training standardized space 

# Query k-NN for every molecule to be labeled (including unlabeled/test); neighbors come from training set only 
d_all, i_all = nn.kneighbors(X_all_std, return_distance=True)  # Shapes: (n_all, k) each 

# Aggregate distances as reliability signals (smaller distance => more in-domain) 
knn_mean_all = d_all.mean(axis=1)  # Mean k-NN distance per query 
knn_median_all = np.median(d_all, axis=1)  # Median k-NN distance per query 

# --------------------------------
# Section B: Global AD decisions
# --------------------------------
# In-domain flags under global quantile cutoffs calibrated on training set (e.g., 95th percentile) 
inAD_mean_q_all = knn_mean_all <= cutoff_mean_q  # Mean-based global AD label 
inAD_median_q_all = knn_median_all <= cutoff_median_q  # Median-based global AD label 

# Continuous reliability scores (lower is better); normalized versions help cross-project interpretation 
rel_mean = knn_mean_all  # Raw mean distance as reliability proxy 
rel_median = knn_median_all  # Raw median distance as reliability proxy 
rel_mean_norm = knn_mean_all / (cutoff_mean_q + 1e-12)  # <=1 typically means in-domain under mean criterion 
rel_median_norm = knn_median_all / (cutoff_median_q + 1e-12)  # <=1 typically means in-domain under median criterion 

# --------------------------------
# Section C: Leverage (Williams) check
# --------------------------------
# Compute leverages for all queries using training (X^T X)^{-1} with intercept; flag extrapolation via h* = 3(p+1)/n 
X_aug_train = np.hstack([np.ones((X_train_std.shape[0], 1)), X_train_std])  # Add intercept for leverage count 
XtX = X_aug_train.T @ X_aug_train  # Cross-product matrix for hat computation 
XtX_inv = np.linalg.pinv(XtX)  # Pseudo-inverse for stability in high-dimensional settings 
X_aug_all = np.hstack([np.ones((X_all_std.shape[0], 1)), X_all_std])  # Augment all queries with intercept 
leverages_all = np.einsum("ij,jk,ik->i", X_aug_all, XtX_inv, X_aug_all)  # diag(X H X^T) without forming full hat matrix 
p_eff = X_aug_train.shape[1]  # Parameters including intercept for h* 
n_train = X_train_std.shape[0]  # Training count for h* 
h_star = 3.0 * p_eff / n_train  # Common Williams threshold used in QSAR AD practice 

inAD_leverage_all = leverages_all <= h_star  # True if within leverage warning threshold 

# --------------------------------
# Section D: Combine, attach, and save
# --------------------------------
# Combine mean-distance and leverage checks as a conservative final AD label (AND rule); adjust per project policy 
inAD_final_all = inAD_mean_q_all & inAD_leverage_all  # Both neighborhood and leverage must be acceptable 

# Package outputs in a DataFrame aligned to df_all if available; else index from range(n_all) 
index_all = df_all.index if 'df_all' in globals() else pd.RangeIndex(X_all_std.shape[0])  # Preserve original indexing if present 
ad_all = pd.DataFrame({
    "k": k,  # Stored for traceability in reports/QPRF 
    "metric": metric,  # Distance metric used 
    "kNN_mean": knn_mean_all,  # Mean k-NN distance to training 
    "kNN_median": knn_median_all,  # Median k-NN distance to training 
    "cutoff_mean_q": cutoff_mean_q,  # Global mean-distance cutoff (e.g., 95th pct) 
    "cutoff_median_q": cutoff_median_q,  # Global median-distance cutoff (e.g., 95th pct) 
    "inAD_mean_q": inAD_mean_q_all,  # In-AD flag by mean-distance criterion 
    "inAD_median_q": inAD_median_q_all,  # In-AD flag by median-distance criterion 
    "leverage_h": leverages_all,  # Williams leverage for extrapolation diagnostics 
    "h_star": h_star,  # Warning leverage threshold 
    "inAD_leverage": inAD_leverage_all,  # In-AD by leverage 
    "rel_mean": rel_mean,  # Continuous reliability score (lower is better) 
    "rel_median": rel_median,  # Continuous reliability score (lower is better) 
    "rel_mean_norm": rel_mean_norm,  # Normalized by cutoff (<=1 ~ in-domain) 
    "rel_median_norm": rel_median_norm,  # Normalized by cutoff (<=1 ~ in-domain) 
    "inAD_final": inAD_final_all  # Conservative final AD decision (mean + leverage) 
}, index=index_all)  # Align to original df_all rows for straightforward joins 

# Optionally attach to df_all for export and downstream filtering 
if 'df_all' in globals():
    df_all_with_ad = pd.concat([df_all.reset_index(drop=True), ad_all.reset_index(drop=True)], axis=1)  # Merge features/meta with AD 
    df_all_with_ad.to_csv("ad_labels_all.csv", index=False)  # Persist AD labels for auditability and screening 

# Print a brief summary
n_all = X_all_std.shape[0]  # Total molecules labeled by AD 
print(f"AD labeling complete: n_all={n_all}, inAD_mean_q={inAD_mean_q_all.sum()}, inAD_final={inAD_final_all.sum()}")  # Quick counts for sanity check 


AD labeling complete: n_all=781179, inAD_mean_q=680444, inAD_final=672013


In [19]:
df_all_with_ad.head()

,number,SMILES,Name,Affinity,label,MaxEStateIndex,MinEStateIndex,MinAbsEStateIndex,qed,MolWt,...,inAD_mean_q,inAD_median_q,leverage_h,h_star,inAD_leverage,rel_mean,rel_median,rel_mean_norm,rel_median_norm,inAD_final
0,1,N=c1nc2n(cc1F)[C@H]1O[C@H](CO)[C@@H](O)[C@H]1O2,ZINC000000001477,-6.1,1,13.213502,-1.010236,0.029481,0.557798,243.194,...,True,True,0.017916,0.035439,True,5.941530,7.740716,0.511026,0.630280,True
1,2,Cc1cn([C@H]2C[C@@H](CO)N(O)C2)c(=O)[nH]c1=O,ZINC000000004266,-5.9,0,11.625685,-0.488343,0.162613,0.606489,241.247,...,True,True,0.018649,0.035439,True,7.582958,7.437727,0.652204,0.605610,True
2,3,CC(=O)N[C@@H]1C[C@@H](O)[C@H](CO)O[C@@H]1O,ZINC000000005637,-5.0,0,10.706467,-1.181065,0.172593,0.415739,205.210,...,True,True,0.002091,0.035439,True,2.648957,3.256916,0.227835,0.265191,True
3,4,Nc1ncc2c(ncn2COC(CO)CO)n1,ZINC000000005980,-5.3,0,8.853217,-0.609120,0.148378,0.595031,239.235,...,True,True,0.010673,0.035439,True,6.151045,6.415066,0.529047,0.522340,True
4,5,Cc1cn([C@H]2O[C@@H](CO)[C@@H]2CO)c(=O)[nH]c1=O,ZINC000000006018,-5.7,0,11.566241,-0.638704,0.193941,0.588358,242.231,...,True,True,0.006201,0.035439,True,3.570982,3.766010,0.307137,0.306644,True


In [18]:
df_all_with_ad.tail()

,number,SMILES,Name,Affinity,label,MaxEStateIndex,MinEStateIndex,MinAbsEStateIndex,qed,MolWt,...,inAD_mean_q,inAD_median_q,leverage_h,h_star,inAD_leverage,rel_mean,rel_median,rel_mean_norm,rel_median_norm,inAD_final
781174,782193,COC[C@@H]1[C@H](NC(CO)CO)[C@@H]2CCO[C@H]12,ZINC000218404185,NaN,0,9.066176,-0.230642,0.036760,0.545618,231.292,...,True,True,0.010399,0.035439,True,8.559852,8.673387,0.736226,0.706222,True
781175,782194,CN(C)CCO[C@@H]1COCC[C@H]1NC(=O)CO,ZINC000218743468,NaN,0,11.141895,-0.484350,0.064294,0.617836,246.307,...,True,True,0.009340,0.035439,True,8.777544,8.854157,0.754950,0.720941,True
781176,782195,Cn1cc(S(=O)(=O)F)c(=O)n(C)c1=O,ZINC000238857165,NaN,0,12.517260,-5.088310,0.520301,0.555618,222.197,...,True,True,0.045648,0.035439,False,11.012425,11.083697,0.947170,0.902479,False
781177,782196,O=C1NC(=O)[C@@H](CCS(=O)(=O)F)N1,ZINC000307689379,NaN,0,11.994428,-4.583802,0.253565,0.455746,210.186,...,True,True,0.030270,0.035439,True,9.000753,9.252645,0.774148,0.753388,True
781178,782197,O=C1NC(=O)[C@H](CCS(=O)(=O)F)N1,ZINC000307689380,NaN,0,11.994428,-4.583802,0.253565,0.455746,210.186,...,True,True,0.030270,0.035439,True,9.000753,9.252645,0.774148,0.753388,True


In [22]:
# =========================
# PHASE 4: VALIDATE AND REPORT WITH AD (robust merge; fixes InvalidIndexError from non-unique SMILES)
# - Aggregates predictions by SMILES to ensure unique keys
# - Inserts Predicted_Prob and Predicted_Label at columns 6 and 7 (1-indexed)
# - Computes metrics on labeled subset overall, in-AD, and out-of-AD
# =========================

import numpy as np                              # Numeric operations
import pandas as pd                             # Data handling
from sklearn.metrics import (                   # Metrics
    average_precision_score, roc_auc_score, accuracy_score,
    f1_score, precision_score, recall_score, brier_score_loss
)
from sklearn.calibration import calibration_curve  # Reliability curve data

# -----------------------------------------
# Section A: Load/prepare frames
# -----------------------------------------
# df_all_with_ad must exist from Phase 3; if not, build it from base + saved AD
if 'df_all_with_ad' not in globals():
    base_df = pd.read_csv("class_df_all_with_filtered_rdkit_features.csv")  # Base input
    ad_df = pd.read_csv("ad_labels_all.csv")                                # From Phase 3
    if len(base_df) != len(ad_df):
        raise ValueError("Row count mismatch between base data and AD labels.")
    df_all_with_ad = pd.concat([base_df.reset_index(drop=True),
                                ad_df.reset_index(drop=True)], axis=1)

# Load predictions file (may contain duplicate SMILES)
pred_path = "full_classification_predictions.csv"
pred_df = pd.read_csv(pred_path)

# Ensure required columns exist in predictions
for c in ["SMILES", "Predicted_Prob", "Predicted_Label"]:
    if c not in pred_df.columns:
        raise ValueError(f"Missing '{c}' in predictions file: {pred_path}")

# Coerce types safely
pred_df["Predicted_Prob"] = pd.to_numeric(pred_df["Predicted_Prob"], errors="coerce")  # Probabilities
pred_df["Predicted_Label"] = pd.to_numeric(pred_df["Predicted_Label"], errors="coerce")  # 0/1 labels

# -----------------------------------------
# Section B: Aggregate predictions by SMILES (unique keys to avoid InvalidIndexError)
# -----------------------------------------
# Strategy:
# - Predicted_Prob: mean over duplicates
# - Predicted_Label: majority via rounded mean (ties -> 1 if mean==0.5 due to round-half-up)
pred_agg = (
    pred_df.groupby("SMILES", as_index=False)
           .agg(
               Predicted_Prob=("Predicted_Prob", "mean"),
               Predicted_Label=("Predicted_Label", lambda s: int(round(pd.to_numeric(s, errors="coerce").fillna(0).mean())))
           )
)

# Build unique Series mappers (indexes are now unique)
prob_map = pred_agg.set_index("SMILES")["Predicted_Prob"]
plab_map = pred_agg.set_index("SMILES")["Predicted_Label"]

# -----------------------------------------
# Section C: Insert Predicted_Prob/Predicted_Label at columns 6 and 7 (1-indexed)
# -----------------------------------------
# Map by SMILES preserving df_all_with_ad row order; avoid InvalidIndexError by using unique index Series
if "SMILES" not in df_all_with_ad.columns:
    raise ValueError("df_all_with_ad is missing 'SMILES' column required for merging predictions.")

# Create mapped columns
mapped_prob = df_all_with_ad["SMILES"].map(prob_map)
mapped_plab = df_all_with_ad["SMILES"].map(plab_map)

# Drop existing columns if present to avoid duplicates before inserting
for c in ["Predicted_Prob", "Predicted_Label"]:
    if c in df_all_with_ad.columns:
        df_all_with_ad = df_all_with_ad.drop(columns=[c])

# Insert at exact positions: 6th and 7th columns (1-indexed) => positions 5 and 6 (0-indexed)
insert_pos_prob = 5
insert_pos_plab = 6
df_all_with_ad.insert(insert_pos_prob, "Predicted_Prob", mapped_prob.values)
df_all_with_ad.insert(insert_pos_plab, "Predicted_Label", mapped_plab.values)

# Optional: persist merged frame for auditability
df_all_with_ad.to_csv("full_with_ad_and_predictions.csv", index=False)

# -----------------------------------------
# Section D: Coverage (all rows) and labeled subset selection
# -----------------------------------------
coverage_overall = df_all_with_ad["inAD_final"].mean()                    # Fraction in-AD across all
n_all = len(df_all_with_ad)                                               # Total rows
n_inAD_all = int(df_all_with_ad["inAD_final"].sum())                      # In-AD count
print(f"Coverage (all): {coverage_overall:.4f}  | In-AD (all): {n_inAD_all}/{n_all}")

# Labeled subset requires Affinity notna and a ground-truth label column ('Label' or 'label')
if "Affinity" not in df_all_with_ad.columns:
    raise ValueError("Affinity column not found in df_all_with_ad.")
labeled_mask = df_all_with_ad["Affinity"].notna()
df_lab = df_all_with_ad.loc[labeled_mask].copy()

# Determine ground-truth label column name
true_label_col = "Label" if "Label" in df_lab.columns else ("label" if "label" in df_lab.columns else None)
if true_label_col is None:
    raise ValueError("Ground-truth label column not found (expected 'Label' or 'label').")

# Ensure predictions exist for labeled rows
if df_lab["Predicted_Prob"].isna().any() or df_lab["Predicted_Label"].isna().any():
    missing_n = int(df_lab["Predicted_Prob"].isna().sum() + df_lab["Predicted_Label"].isna().sum())
    raise ValueError(f"Missing Predicted_Prob/Predicted_Label for some labeled rows after merge (missing count ~ {missing_n}).")

# Extract arrays
y_true = df_lab[true_label_col].astype(int).values
y_prob = df_lab["Predicted_Prob"].astype(float).values
y_pred = df_lab["Predicted_Label"].astype(int).values
inAD_mask = df_lab["inAD_final"].values
outAD_mask = ~inAD_mask

n_lab = len(df_lab)
n_inAD_lab = int(inAD_mask.sum())
coverage_labeled = inAD_mask.mean() if n_lab else np.nan
print(f"Coverage (labeled): {coverage_labeled:.4f} | In-AD (labeled): {n_inAD_lab}/{n_lab}")

# -----------------------------------------
# Section E: Helpers for safe AUC/AP
# -----------------------------------------
def safe_auc(y, s):
    return roc_auc_score(y, s) if len(np.unique(y)) > 1 else np.nan  # ROC-AUC only if both classes present

def safe_ap(y, s):
    return average_precision_score(y, s) if len(np.unique(y)) > 1 else np.nan  # PR-AUC only if both classes present

# -----------------------------------------
# Section F: Metrics — overall labeled
# -----------------------------------------
ap_all = safe_ap(y_true, y_prob)                           # PR-AUC
roc_all = safe_auc(y_true, y_prob)                         # ROC-AUC
acc_all = accuracy_score(y_true, y_pred)                   # Accuracy
f1_all = f1_score(y_true, y_pred, zero_division=0)         # F1
prec_all = precision_score(y_true, y_pred, zero_division=0)# Precision
rec_all = recall_score(y_true, y_pred, zero_division=0)    # Recall
brier_all = brier_score_loss(y_true, y_prob)               # Brier score
pm_all, fp_all = calibration_curve(y_true, y_prob, n_bins=10, strategy="uniform")  # Reliability curve data
ece_all = np.average(np.abs(pm_all - fp_all), weights=np.full_like(pm_all, 1/len(pm_all)))  # Simple ECE

# -----------------------------------------
# Section G: Metrics — in-AD labeled
# -----------------------------------------
y_true_in, y_prob_in, y_pred_in = y_true[inAD_mask], y_prob[inAD_mask], y_pred[inAD_mask]
ap_in = safe_ap(y_true_in, y_prob_in)
roc_in = safe_auc(y_true_in, y_prob_in)
acc_in = accuracy_score(y_true_in, y_pred_in) if len(y_true_in) else np.nan
f1_in = f1_score(y_true_in, y_pred_in, zero_division=0) if len(y_true_in) else np.nan
prec_in = precision_score(y_true_in, y_pred_in, zero_division=0) if len(y_true_in) else np.nan
rec_in = recall_score(y_true_in, y_pred_in, zero_division=0) if len(y_true_in) else np.nan
if len(y_true_in) >= 10:
    pm_in, fp_in = calibration_curve(y_true_in, y_prob_in, n_bins=10, strategy="uniform")
    ece_in = np.average(np.abs(pm_in - fp_in), weights=np.full_like(pm_in, 1/len(pm_in)))
else:
    brier_in, ece_in = (np.nan, np.nan)
brier_in = brier_score_loss(y_true_in, y_prob_in) if len(y_true_in) else np.nan

# -----------------------------------------
# Section H: Metrics — out-of-AD labeled
# -----------------------------------------
y_true_out, y_prob_out, y_pred_out = y_true[outAD_mask], y_prob[outAD_mask], y_pred[outAD_mask]
ap_out = safe_ap(y_true_out, y_prob_out)
roc_out = safe_auc(y_true_out, y_prob_out)
acc_out = accuracy_score(y_true_out, y_pred_out) if len(y_true_out) else np.nan
f1_out = f1_score(y_true_out, y_pred_out, zero_division=0) if len(y_true_out) else np.nan
prec_out = precision_score(y_true_out, y_pred_out, zero_division=0) if len(y_true_out) else np.nan
rec_out = recall_score(y_true_out, y_pred_out, zero_division=0) if len(y_true_out) else np.nan
if len(y_true_out) >= 10:
    pm_out, fp_out = calibration_curve(y_true_out, y_prob_out, n_bins=10, strategy="uniform")
    ece_out = np.average(np.abs(pm_out - fp_out), weights=np.full_like(pm_out, 1/len(pm_out)))
else:
    brier_out, ece_out = (np.nan, np.nan)
brier_out = brier_score_loss(y_true_out, y_prob_out) if len(y_true_out) else np.nan

# -----------------------------------------
# Section I: Assemble and save report
# -----------------------------------------
rows = [
    {"subset": "labeled_all",   "n": int(n_lab),
     "PR_AUC": ap_all,   "ROC_AUC": roc_all,   "ACC": acc_all,   "F1": f1_all,
     "Precision": prec_all,     "Recall": rec_all,               "Brier": brier_all, "ECE_10": float(ece_all)},
    {"subset": "labeled_inAD",  "n": int(len(y_true_in)),
     "PR_AUC": ap_in,    "ROC_AUC": roc_in,    "ACC": acc_in,    "F1": f1_in,
     "Precision": prec_in,      "Recall": rec_in,                "Brier": brier_in,
     "ECE_10": float(ece_in) if not np.isnan(ece_in) else np.nan},
    {"subset": "labeled_outAD", "n": int(len(y_true_out)),
     "PR_AUC": ap_out,   "ROC_AUC": roc_out,   "ACC": acc_out,   "F1": f1_out,
     "Precision": prec_out,     "Recall": rec_out,               "Brier": brier_out,
     "ECE_10": float(ece_out) if not np.isnan(ece_out) else np.nan},
]
ad_validation_report = pd.DataFrame(rows)
ad_validation_report.insert(1, "coverage_all", coverage_overall)
ad_validation_report.insert(2, "coverage_labeled", coverage_labeled)

print("\nAD Validation Report (labeled subsets):")
print(ad_validation_report.round(4))

ad_validation_report.to_csv("ad_phase4_validation_report.csv", index=False)


Coverage (all): 0.8603  | In-AD (all): 672013/781179
Coverage (labeled): 0.9472 | In-AD (labeled): 9463/9990

AD Validation Report (labeled subsets):
          subset  coverage_all  coverage_labeled     n  PR_AUC  ROC_AUC  \
0    labeled_all        0.8603            0.9472  9990  0.6724   0.9170   
1   labeled_inAD        0.8603            0.9472  9463  0.6361   0.9125   
2  labeled_outAD        0.8603            0.9472   527  0.8567   0.9243   

      ACC      F1  Precision  Recall   Brier  ECE_10  
0  0.8720  0.6046     0.5044  0.7546  0.1194  0.2718  
1  0.8740  0.5788     0.4787  0.7319  0.1180  0.2734  
2  0.8349  0.7852     0.6974  0.8983  0.1444  0.2468  


In [23]:
# =========================
# Create top-20 high-probability active compounds CSV for presentation
# - Uses df_all_with_ad produced in Phase 3 and merged with predictions in Phase 4
# - Filters to in-AD and Predicted_Label==1 for high-confidence actives
# - Sorts by Predicted_Prob desc and writes required columns
# =========================

import pandas as pd  # Data handling
import numpy as np   # Numeric ops

# Ensure df_all_with_ad exists; if not, load the file created previously
if 'df_all_with_ad' not in globals():
    df_all_with_ad = pd.read_csv("full_with_ad_and_predictions.csv")

# Verify required columns
required_cols = ["number", "SMILES", "Name", "Affinity", "Predicted_Prob", "Predicted_Label", "inAD_final"]
missing = [c for c in required_cols if c not in df_all_with_ad.columns]
if missing:
    raise ValueError(f"Missing required columns in df_all_with_ad: {missing}")

# Conservative presentation set: in-AD and predicted active, ranked by probability
present_cols = ["number", "SMILES", "Name", "Affinity", "Predicted_Prob", "Predicted_Label", "inAD_final"]
mask_inAD = df_all_with_ad["inAD_final"] == True
mask_pred_active = df_all_with_ad["Predicted_Label"].astype(int) == 1

top20_inAD = (
    df_all_with_ad.loc[mask_inAD & mask_pred_active, present_cols]
                  .sort_values("Predicted_Prob", ascending=False)
                  .head(20)
                  .reset_index(drop=True)
)

# Save to CSV
top20_inAD.to_csv("top20_prob_actives_inAD.csv", index=False)

# Optional: also prepare an out-of-AD list to guide Phase 5 active learning triage
top20_outAD = (
    df_all_with_ad.loc[(~mask_inAD) & mask_pred_active, present_cols]
                  .sort_values("Predicted_Prob", ascending=False)
                  .head(20)
                  .reset_index(drop=True)
)
top20_outAD.to_csv("top20_prob_actives_outAD.csv", index=False)

print("Saved: top20_prob_actives_inAD.csv (presentation) and top20_prob_actives_outAD.csv (for AL triage).")


Saved: top20_prob_actives_inAD.csv (presentation) and top20_prob_actives_outAD.csv (for AL triage).


In [30]:
# =========================
# Create a single top-20 CSV of highest-probability predicted actives
# Columns: number, SMILES, Name, Affinity, Predicted_Prob, Predicted_Label, inAD_final
# - Uses df_all_with_ad if present; otherwise loads the merged file from Phase 4
# - Selects Predicted_Label == 1, sorts by Predicted_Prob desc, takes top 20
# =========================

import pandas as pd  # Data handling
import numpy as np   # Numeric ops

# Load source DataFrame
if 'df_all_with_ad' not in globals():
    df_all_with_ad = pd.read_csv("full_with_ad_and_predictions.csv")  # Ensure predictions and AD columns exist

# Required columns
cols_needed = ["number", "SMILES", "Name", "Affinity", "Predicted_Prob", "Predicted_Label", "inAD_final"]
missing = [c for c in cols_needed if c not in df_all_with_ad.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Filter predicted actives and rank by probability
mask_active = df_all_with_ad["Predicted_Label"].astype(int) == 1
top20 = (
    df_all_with_ad.loc[mask_active, cols_needed]
                  .sort_values("Predicted_Prob", ascending=False)
                  .head(500)
                  .reset_index(drop=True)
)

# Save a single CSV with inAD_final included
top20.to_csv("top500_prob_actives.csv", index=False)

print("Saved: top500_prob_actives.csv")


Saved: top500_prob_actives.csv


In [ ]:
# =========================
# Merge AD columns into regression file and export top lists
# Assumptions:
# - df_all_with_predictions.csv contains regression predictions and SMILES/metadata
# - ad_labels_all.csv contains AD columns computed previously (same molecules, same scaling)
# - Sorting for "highest affinity regression" uses the regression prediction column (not the true Affinity)
# =========================

import pandas as pd  # Data handling
import numpy as np   # Numerical ops

# -----------------------------
# Section 1 — File paths
# -----------------------------
REG_CSV = "df_all_with_predictions.csv"            # Regression dataset with predictions
AD_CSV  = "ad_labels_all.csv"                      # AD columns computed previously
OUT_MERGED = "df_reg_with_predictions_with_AD.csv" # Output merged file
OUT_TOP500 = "top500_predicted_affinity_with_inAD.csv"   # Top-500 by predicted value
OUT_TOP20_INAD = "top20_inAD_predicted_affinity.csv"     # Top-20 by predicted value within in-AD

# -----------------------------
# Section 2 — Load data
# -----------------------------
df_reg = pd.read_csv(REG_CSV)  # Load regression predictions
ad_df = pd.read_csv(AD_CSV)    # Load AD annotations

# -----------------------------
# Section 3 — Prepare AD frame
# -----------------------------
# Ensure AD frame has SMILES and select AD columns present
if "SMILES" not in ad_df.columns:
    raise ValueError("ad_labels_all.csv must contain a 'SMILES' column.")

ad_cols_expected = [
    "k", "metric",
    "kNN_mean", "kNN_median",
    "cutoff_mean_q", "cutoff_median_q",
    "inAD_mean_q", "inAD_median_q",
    "leverage_h", "h_star", "inAD_leverage",
    "rel_mean", "rel_median", "rel_mean_norm", "rel_median_norm",
    "inAD_final"
]
ad_cols_present = ["SMILES"] + [c for c in ad_cols_expected if c in ad_df.columns]

# If duplicates in SMILES exist, keep the first occurrence (AD should be identical for identical features)
ad_unique = (
    ad_df[ad_cols_present]
    .drop_duplicates(subset="SMILES", keep="first")
    .reset_index(drop=True)
)

# -----------------------------
# Section 4 — Merge into regression file
# -----------------------------
if "SMILES" not in df_reg.columns:
    raise ValueError("df_all_with_predictions.csv must contain a 'SMILES' column.")

df_merged = df_reg.merge(ad_unique, on="SMILES", how="left")  # Left-join to retain all regression rows
df_merged.to_csv(OUT_MERGED, index=False)                     # Save merged file
print(f"Saved merged file with AD columns: {OUT_MERGED}")

# -----------------------------
# Section 5 — Identify prediction column
# -----------------------------
# Try common prediction column names used in regression outputs
pred_candidates = ["Predicted", "Predicted_Value", "y_pred", "Prediction", "Pred", "Predicted Affinity"]
pred_cols_found = [c for c in pred_candidates if c in df_merged.columns]
if not pred_cols_found:
    raise ValueError(f"No regression prediction column found among {pred_candidates}.")
pred_col = pred_cols_found[0]  # Use the first found



Saved merged file with AD columns: df_reg_with_predictions_with_AD.csv
Saved: top500_predicted_affinity_with_inAD.csv
Saved: top20_inAD_predicted_affinity.csv


In [2]:
df_reg_AD = pd.read_csv("df_reg_with_predictions_with_AD.csv")
df_reg_AD.head()


,number,SMILES,Name,label,Predicted Affinity,MaxEStateIndex,MinEStateIndex,MinAbsEStateIndex,qed,MolWt,...,inAD_mean_q,inAD_median_q,leverage_h,h_star,inAD_leverage,rel_mean,rel_median,rel_mean_norm,rel_median_norm,inAD_final
0,1,N=c1nc2n(cc1F)[C@H]1O[C@H](CO)[C@@H](O)[C@H]1O2,ZINC000000001477,-6.1,-5.980616,13.213502,-1.010236,0.029481,0.557798,243.194,...,True,True,0.017916,0.035439,True,5.941530,7.740716,0.511026,0.630280,True
1,2,Cc1cn([C@H]2C[C@@H](CO)N(O)C2)c(=O)[nH]c1=O,ZINC000000004266,-5.9,-5.900323,11.625685,-0.488343,0.162613,0.606489,241.247,...,True,True,0.018649,0.035439,True,7.582958,7.437727,0.652204,0.605610,True
2,3,CC(=O)N[C@@H]1C[C@@H](O)[C@H](CO)O[C@@H]1O,ZINC000000005637,-5.0,-5.252120,10.706467,-1.181065,0.172593,0.415739,205.210,...,True,True,0.002091,0.035439,True,2.648957,3.256916,0.227835,0.265191,True
3,4,Nc1ncc2c(ncn2COC(CO)CO)n1,ZINC000000005980,-5.3,-5.561441,8.853217,-0.609120,0.148378,0.595031,239.235,...,True,True,0.010673,0.035439,True,6.151045,6.415066,0.529047,0.522340,True
4,5,Cc1cn([C@H]2O[C@@H](CO)[C@@H]2CO)c(=O)[nH]c1=O,ZINC000000006018,-5.7,-5.701878,11.566241,-0.638704,0.193941,0.588358,242.231,...,True,True,0.006201,0.035439,True,3.570982,3.766010,0.307137,0.306644,True


In [3]:
df_reg_AD.tail()

,number,SMILES,Name,label,Predicted Affinity,MaxEStateIndex,MinEStateIndex,MinAbsEStateIndex,qed,MolWt,...,inAD_mean_q,inAD_median_q,leverage_h,h_star,inAD_leverage,rel_mean,rel_median,rel_mean_norm,rel_median_norm,inAD_final
781174,782193,COC[C@@H]1[C@H](NC(CO)CO)[C@@H]2CCO[C@H]12,ZINC000218404185,NaN,-5.147846,9.066176,-0.230642,0.036760,0.545618,231.292,...,True,True,0.010399,0.035439,True,8.559852,8.673387,0.736226,0.706222,True
781175,782194,CN(C)CCO[C@@H]1COCC[C@H]1NC(=O)CO,ZINC000218743468,NaN,-5.087768,11.141895,-0.484350,0.064294,0.617836,246.307,...,True,True,0.009340,0.035439,True,8.777544,8.854157,0.754950,0.720941,True
781176,782195,Cn1cc(S(=O)(=O)F)c(=O)n(C)c1=O,ZINC000238857165,NaN,-5.307266,12.517260,-5.088310,0.520301,0.555618,222.197,...,True,True,0.045648,0.035439,False,11.012425,11.083697,0.947170,0.902479,False
781177,782196,O=C1NC(=O)[C@@H](CCS(=O)(=O)F)N1,ZINC000307689379,NaN,-5.328358,11.994428,-4.583802,0.253565,0.455746,210.186,...,True,True,0.030270,0.035439,True,9.000753,9.252645,0.774148,0.753388,True
781178,782197,O=C1NC(=O)[C@H](CCS(=O)(=O)F)N1,ZINC000307689380,NaN,-5.328358,11.994428,-4.583802,0.253565,0.455746,210.186,...,True,True,0.030270,0.035439,True,9.000753,9.252645,0.774148,0.753388,True


In [3]:
# -----------------------------
# Section 6 — Build output subsets
# -----------------------------
# Required presentation columns
present_cols = ["number", "SMILES", "Name", "Affinity", pred_col, "inAD_final"]

# Ensure required metadata columns exist; if not, create placeholders
for c in ["number", "Name", "Affinity"]:
    if c not in df_merged.columns:
        df_merged[c] = np.nan  # Fill missing optional columns with NaN

# Top-500 by predicted value (descending), include inAD_final
top500 = (
    df_merged
    .sort_values(pred_col, kind="mergesort")  # Stable sort
    .loc[:, present_cols]
    .head(500)
    .reset_index(drop=True)
)

# Top-20 in-AD by predicted value
mask_inAD = df_merged["inAD_final"] == True
top20_inAD = (
    df_merged.loc[mask_inAD, present_cols]
             .sort_values(pred_col, kind="mergesort")
             .head(20)
             .reset_index(drop=True)
)

# -----------------------------
# Section 7 — Save outputs
# -----------------------------
top500.to_csv(OUT_TOP500, index=False)
top20_inAD.to_csv(OUT_TOP20_INAD, index=False)
print(f"Saved: {OUT_TOP500}")
print(f"Saved: {OUT_TOP20_INAD}")


Saved: top500_predicted_affinity_with_inAD.csv
Saved: top20_inAD_predicted_affinity.csv


In [ ]:
# =========================
# Compile regression + classification labels into one file with specified columns/positions
# - Base file: df_reg_with_predictions_with_AD.csv
#   • 'label' = docking affinity scores (true), 'Predicted affinity' = regression predictions
# - AD file: ad_labels_all.csv
#   • 'Affinity' = docking affinity scores (true), 'label' = classification predicted activities
# - Output: ad_reg_cls_features.csv
#   • col4: dock_true_affinity
#   • col5: pre_reg_affinity
#   • col6: pre_cls_activity
# - Preserve row order and the original base columns' order; only add the 3 new columns
# =========================

import pandas as pd  # Data handling
import numpy as np   # Numeric ops

# -----------------------------
# Section 1 — Load inputs
# -----------------------------
base_path = "df_reg_with_predictions_with_AD.csv"   # Base regression file
ad_path = "ad_labels_all.csv"                       # AD/classification file
out_path = "ad_reg_cls_features.csv"                # Output file

df_base = pd.read_csv(base_path)                    # Load base file (order preserved)
df_ad = pd.read_csv(ad_path)                        # Load AD file (used only to map values)

# -----------------------------
# Section 2 — Validate keys and locate source columns
# -----------------------------
# Ensure join key exists in both
if "SMILES" not in df_base.columns or "SMILES" not in df_ad.columns:
    raise ValueError("Both input files must contain a 'SMILES' column for alignment.")

# Regression prediction column candidates in base
reg_pred_candidates = ["Predicted Affinity", "Predicted", "Predicted_Value", "y_pred", "Prediction", "Pred"]
reg_pred_cols_found = [c for c in reg_pred_candidates if c in df_base.columns]
if not reg_pred_cols_found:
    raise ValueError(f"No regression prediction column found in base among {reg_pred_candidates}.")
reg_pred_col = reg_pred_cols_found[0]  # Use the first match

# Base true docking affinity column (per user: base 'label' stores docking affinity)
base_dock_col = "label" if "label" in df_base.columns else None

# AD true docking affinity column
ad_aff_col = "Affinity" if "Affinity" in df_ad.columns else None

# AD classification activity candidates
cls_act_candidates = ["label", "Label", "Predicted_Label", "PredictedLabel", "Predicted_Class",
                      "Predicted_Activity", "PredictedActivity", "class", "Class", "Activity", "activity"]
cls_act_cols_found = [c for c in cls_act_candidates if c in df_ad.columns]
cls_act_col = cls_act_cols_found[0] if cls_act_cols_found else None

# -----------------------------
# Section 3 — Build mapping Series from AD file (unique by SMILES)
# -----------------------------
ad_unique = df_ad.drop_duplicates(subset="SMILES", keep="first").set_index("SMILES")

# Map AD true docking affinity by SMILES (may be missing)
ad_aff_series = ad_unique[ad_aff_col] if ad_aff_col else pd.Series(dtype=float)

# Map AD classification activity by SMILES (may be missing)
ad_cls_series = ad_unique[cls_act_col] if cls_act_col else pd.Series(dtype=float)

# -----------------------------
# Section 4 — Create the three compiled columns
# -----------------------------
# dock_true_affinity: prefer AD 'Affinity' when available; else fall back to base 'label' (docking affinity)
dock_from_ad = df_base["SMILES"].map(ad_aff_series) if ad_aff_col else pd.Series([np.nan]*len(df_base))
dock_from_base = df_base[base_dock_col] if base_dock_col else pd.Series([np.nan]*len(df_base))
dock_true_affinity = dock_from_ad.where(~dock_from_ad.isna(), dock_from_base)

# pre_reg_affinity: regression predicted affinities from base
pre_reg_affinity = df_base[reg_pred_col]

# pre_cls_activity: classification predicted activities from AD; if not found, fill NaN
pre_cls_activity = df_base["SMILES"].map(ad_cls_series) if cls_act_col else pd.Series([np.nan]*len(df_base))

# -----------------------------
# Section 5 — Construct output with preserved base order + inserted columns
# -----------------------------
df_out = df_base.copy()  # Start from base to keep row and column order

# Drop if these columns already exist to avoid duplicates
for c in ["dock_true_affinity", "pre_reg_affinity", "pre_cls_activity"]:
    if c in df_out.columns:
        df_out = df_out.drop(columns=[c])

# Insert at positions 4–6 (1-indexed) => indices 3, 4, 5 (0-indexed)
df_out.insert(3, "dock_true_affinity", pd.to_numeric(dock_true_affinity, errors="coerce").values)
df_out.insert(4, "pre_reg_affinity", pd.to_numeric(pre_reg_affinity, errors="coerce").values)
df_out.insert(5, "pre_cls_activity", pd.to_numeric(pre_cls_activity, errors="coerce").values)

# Drop unwanted columns from df_out and resave

# Remove columns if present; ignore if already absent
df_out = df_out.drop(columns=["Predicted Affinity", "label"], errors="ignore")

# -----------------------------
# Section 6 — Save output
# -----------------------------
df_out.to_csv(out_path, index=False)
print(f"Saved: {out_path}")

# -----------------------------
# Optional: quick sanity prints (commented)
# -----------------------------
# print(df_out.iloc[:2, :12])


Saved: ad_reg_cls_features.csv


In [11]:
# Drop unwanted columns from df_out and resave

# Remove columns if present; ignore if already absent
df_out = df_out.drop(columns=["Predicted Affinity", "label"], errors="ignore")

# Save back to the same file
df_out.to_csv("ad_reg_cls_features.csv", index=False)


In [12]:
df_out.head()

,number,SMILES,Name,dock_true_affinity,pre_reg_affinity,pre_cls_activity,MaxEStateIndex,MinEStateIndex,MinAbsEStateIndex,qed,...,inAD_mean_q,inAD_median_q,leverage_h,h_star,inAD_leverage,rel_mean,rel_median,rel_mean_norm,rel_median_norm,inAD_final
0,1,N=c1nc2n(cc1F)[C@H]1O[C@H](CO)[C@@H](O)[C@H]1O2,ZINC000000001477,-6.1,-5.980616,1,13.213502,-1.010236,0.029481,0.557798,...,True,True,0.017916,0.035439,True,5.941530,7.740716,0.511026,0.630280,True
1,2,Cc1cn([C@H]2C[C@@H](CO)N(O)C2)c(=O)[nH]c1=O,ZINC000000004266,-5.9,-5.900323,0,11.625685,-0.488343,0.162613,0.606489,...,True,True,0.018649,0.035439,True,7.582958,7.437727,0.652204,0.605610,True
2,3,CC(=O)N[C@@H]1C[C@@H](O)[C@H](CO)O[C@@H]1O,ZINC000000005637,-5.0,-5.252120,0,10.706467,-1.181065,0.172593,0.415739,...,True,True,0.002091,0.035439,True,2.648957,3.256916,0.227835,0.265191,True
3,4,Nc1ncc2c(ncn2COC(CO)CO)n1,ZINC000000005980,-5.3,-5.561441,0,8.853217,-0.609120,0.148378,0.595031,...,True,True,0.010673,0.035439,True,6.151045,6.415066,0.529047,0.522340,True
4,5,Cc1cn([C@H]2O[C@@H](CO)[C@@H]2CO)c(=O)[nH]c1=O,ZINC000000006018,-5.7,-5.701878,0,11.566241,-0.638704,0.193941,0.588358,...,True,True,0.006201,0.035439,True,3.570982,3.766010,0.307137,0.306644,True


In [13]:
df_out.tail()

,number,SMILES,Name,dock_true_affinity,pre_reg_affinity,pre_cls_activity,MaxEStateIndex,MinEStateIndex,MinAbsEStateIndex,qed,...,inAD_mean_q,inAD_median_q,leverage_h,h_star,inAD_leverage,rel_mean,rel_median,rel_mean_norm,rel_median_norm,inAD_final
781174,782193,COC[C@@H]1[C@H](NC(CO)CO)[C@@H]2CCO[C@H]12,ZINC000218404185,NaN,-5.147846,0,9.066176,-0.230642,0.036760,0.545618,...,True,True,0.010399,0.035439,True,8.559852,8.673387,0.736226,0.706222,True
781175,782194,CN(C)CCO[C@@H]1COCC[C@H]1NC(=O)CO,ZINC000218743468,NaN,-5.087768,0,11.141895,-0.484350,0.064294,0.617836,...,True,True,0.009340,0.035439,True,8.777544,8.854157,0.754950,0.720941,True
781176,782195,Cn1cc(S(=O)(=O)F)c(=O)n(C)c1=O,ZINC000238857165,NaN,-5.307266,0,12.517260,-5.088310,0.520301,0.555618,...,True,True,0.045648,0.035439,False,11.012425,11.083697,0.947170,0.902479,False
781177,782196,O=C1NC(=O)[C@@H](CCS(=O)(=O)F)N1,ZINC000307689379,NaN,-5.328358,0,11.994428,-4.583802,0.253565,0.455746,...,True,True,0.030270,0.035439,True,9.000753,9.252645,0.774148,0.753388,True
781178,782197,O=C1NC(=O)[C@H](CCS(=O)(=O)F)N1,ZINC000307689380,NaN,-5.328358,0,11.994428,-4.583802,0.253565,0.455746,...,True,True,0.030270,0.035439,True,9.000753,9.252645,0.774148,0.753388,True


In [ ]:
# Extract data for external validation
df_out = pd.read_csv("ad_reg_cls_features.csv")

# Define desired columns
subset_cols = ["number", "SMILES", "Name", "dock_true_affinity", "pre_reg_affinity", "pre_cls_activity", "inAD_final" ]  # Target columns

# Quick check for missing columns
missing = [c for c in subset_cols if c not in df_out.columns]  # Find absent columns
if missing:
    raise ValueError(f"Missing required columns in df_out: {missing}")  # Fail fast if any are missing

# Build the subset DataFrame
df_sub = df_out[subset_cols].copy()  # Preserve order and values

# Write to CSV
df_sub.to_csv("df_full_wo_features.csv", index=False)  # Save without index


In [ ]:
# Extract number, SMILES, and dock_true_affinity from df_out and save to CSV

# Define desired columns
subset_cols = ["number", "SMILES", "dock_true_affinity"]  # Target columns

# Quick check for missing columns
missing = [c for c in subset_cols if c not in df_out.columns]  # Find absent columns
if missing:
    raise ValueError(f"Missing required columns in df_out: {missing}")  # Fail fast if any are missing

# Build the subset DataFrame
df_subset = df_out[subset_cols].copy()  # Preserve order and values

# Write to CSV
df_subset.to_csv("smiles.csv", index=False)  # Save without index
